In [1]:
## Uncomment and run the below to commands for reloading the modules of un_llm_analysis after changes. 
# %load_ext autoreload
# %autoreload 2

# Issue Analysis on the UN Daily Briefings

In this analysis, an "issue" refers to a substantive topic or situation discussed in the UN Noon Briefings, such as *the Israeli-Palestinian conflict*, *the war in Ukraine*, *the conflict in Sudan*, or *climate change*. Sections of the UN Noon Briefings are labeled if they are related to any of the selected issues. 

The purpose of the analysis is to examine how the presence of these selected issues in the briefings changes over time.

The analysis consists of two notebooks:
- [Identifying Issues for Analysis of UN Briefings](./01_identify_issues.ipynb) 
- [Issue Analysis on the UN Daily Briefings](./02_issue_proportion_in_briefings.ipynb) 

This notebook uses an LLM to label each section with the relevant issues and visualizes how the frequency and text proportion of the selected issues change over time.

## 1. Set up and load data

This notebook uses the pre-uploaded database specified by `BRIEFINGS_DB_PATH`. 

Alternatively, the database can be created from scratch for a desired date range:
```bash
briefings update 2026-09-01 2026-09-10
briefings init-db
briefings load 2026-09-01 2026-09-10
```

In [2]:
import mlflow

from un_llm_analysis.analysis.issue import (
    get_issue_text,
    SELECTED_ISSUES,
    get_section_data,
    get_issue_data,
    calculate_issue_stats,
    calculate_issue_counts,
    GenerationConfig,
    map_issue_to_sections,
)
from un_llm_analysis.database import (
    get_db_connection, 
    create_database,
)
from un_llm_analysis.prompts import ISSUE_MAPPING_PROMPT

In [3]:
# Create database and make sure the tables are created.
create_database()

# Get the data in the database within the specified date range
start_date = "2026-09-01"
end_date = "2026-09-05"

# Get the section data from the briefings and store it in a DataFrame
section_data = get_section_data(
    start_date=start_date,
    end_date=end_date,
)

section_data.info()
section_data.head(2)

2026-11-11 15:00:16.121 | INFO     | un_llm_analysis.database:create_database:125 - Database created in "/home/yhs/projects/correlaid/UN-LLM-Analysis/app/briefings.db"


<class 'pandas.DataFrame'>
RangeIndex: 49 entries, 0 to 48
Data columns (total 4 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   section_id     49 non-null     int64
 1   briefing_id    49 non-null     int64
 2   section_title  49 non-null     str  
 3   section_text   49 non-null     str  
dtypes: int64(2), str(2)
memory usage: 58.1 KB


,section_id,briefing_id,section_title,section_text
0,2026090101,20260901,TODAY’S BRIFING,"At 12:30 p.m., Jérôme Bonnafont, the Permanent..."
1,2026090102,20260901,TOMORROW’S GUEST,"Inger Andersen, the Executive Director of UNEP..."


## 2. Issue Mapping

Refer to [Identifying Issues for Analysis of UN Briefings](./01_identify_issues.ipynb) for details on how the issues used in the subsequent analysis are identified.

In [4]:
issue_definitions = get_issue_text(SELECTED_ISSUES)
print(issue_definitions)

* ISSUE-1 Israel-Palestine conflict, Gaza and the Occupied Palestinian Territory
* ISSUE-2 War in Ukraine and its direct security and humanitarian consequences
* ISSUE-3 Sudan conflict, political process and humanitarian crisis
* ISSUE-4 Democratic Republic of the Congo conflict and humanitarian crisis
* ISSUE-5 Yemen conflict and humanitarian crisis
* ISSUE-6 South Sudan conflict, peace process and humanitarian crisis
* ISSUE-7 Syria conflict and humanitarian crisis
* ISSUE-8 Haiti political, security and humanitarian crisis
* ISSUE-9 Afghanistan political and humanitarian crisis
* ISSUE-10 Myanmar conflict and Rohingya crisis
* ISSUE-11 Central African Republic conflict and political transition
* ISSUE-12 Lebanon political and humanitarian crisis and Israel-Lebanon hostilities
* ISSUE-13 Ethiopia conflict and humanitarian crisis
* ISSUE-14 Libya political transition and insecurity
* ISSUE-15 Sahel and Lake Chad Basin conflict and insecurity
* ISSUE-16 Somalia conflict and humanitaria

**A comment on the selected issues**

Although we hope the issues can be as specific as possible, it is difficult to avoid including some that are not so well defined or are a little vague. Overall, this is already a relatively balanced list.

From the list, we can see that ISSUE-1 to ISSUE-16 are more specific and related to particular countries, regions, or geographical areas, while ISSUE-17 to ISSUE-24 are somewhat broader and are not directly related to any specific geographical location.

In [5]:
prompt = ISSUE_MAPPING_PROMPT
print(prompt)

**Task**

Identify the most relevant issue for the text below.

**Issue Definitions**

{issue_definitions}

**Text**

{section_text}

**Decision Rules**

- Assign one issue by default.
- Return None if none of the issue definitions clearly apply.
- Assign two issues only if the text contains two distinct substantive topics and each independently matches an issue definition.
- Never assign more than two issues.
- Do not assign a second issue only because it is mentioned as background, consequence, or context.

**Output Format**

Return exactly one of the following forms:

None
['ISSUE-<number>']
['ISSUE-<number>', 'ISSUE-<number>']

Do not return ['None'].
Do not include any explanation or additional text.



### Ollama and mlflow

Set up Ollama and mlflow to run local LLMs and capture detailed tracing records.

Make sure you have synced dependencies with `uv sync`.

```bash
# Download LLMs to test
ollama pull qwen3.5:4b # Other models: llama3.2 gemma3:4b qwen3.5:4b llama3.2:1b qwen3:1.7b

# Start a local mlflow server
uv run mlflow server # Or run it in the background with: `nohup uv run mlflow server --port 5000 > mlflow.log 2>&1 &`
```

In [ ]:
# This cell is the main loop for running the issue mapping experiments with the LLM.

mlflow.openai.autolog() 
mlflow.set_tracking_uri("http://127.0.0.1:5000") 
mlflow.set_experiment("Issue Mapping")

# Here we use `qwen3.5:4b` because it produced the best results for this task among the models tested below.
# models = ["llama3.2:latest", "gemma3:4b", "qwen3.5:4b", "llama3.2:1b", "qwen3:1.7b"]
models = ["qwen3.5:4b"] # More models can be added for comparison if needed.
config = GenerationConfig()

conn = get_db_connection()

for llm in models:
    # Map issues to sections using the specified LLM model and store the results in the database.
    map_issue_to_sections(
        conn=conn, start_date=start_date, end_date=end_date, model=llm, config=config, overwrite=False
    )

conn.close()

## 3. Visualization

### 1) Build the DataFrame for analysis

In [6]:
# Get the issue data from the database within the specified date range
model = "qwen3.5:4b"
all_issues = list(SELECTED_ISSUES.keys())

issue_data = get_issue_data(
    model=model,
    issues=all_issues,
    start_date=start_date,
    end_date=end_date,
)

issue_data.head(2)

,briefing_id,section_id,issue,model,created_at,date
0,20260902,2026090211,ISSUE-17,qwen3.5:4b,2026-09-23T15:02:21+00:00,2026-09-02
1,20260901,2026090102,ISSUE-17,qwen3.5:4b,2026-10-04T22:35:58+00:00,2026-09-01


In [7]:
# Calculate issue lengths, frequencies, and proportions by time period.
plot_level = "day"  # Change to "week" or "month" for different time periods
issue_stats = calculate_issue_stats(
    section_data=section_data,
    issue_data=issue_data,
    level=plot_level,
)

issue_stats.head(2)

,day,issue,total_length,issue_length,frequency,proportion
0,2026-09-01,ISSUE-1,14993,3965,2,0.264457
1,2026-09-01,ISSUE-2,14993,1638,1,0.109251


In [8]:
# Calculate issue_counts from issue_data
issue_counts = calculate_issue_counts(issue_data)

issue_counts.head(2)

,issue,total_counts
0,ISSUE-20,13
1,ISSUE-18,8


### 2) Draw the plots

In [9]:
import plotly.express as px

fig_1 = px.line(
    issue_stats,
    x=plot_level,
    y="frequency",
    color="issue",
    markers=False,
    title=f"Frequency of Issues in UN Briefings between {start_date} and {end_date}"
)

fig_1.show()

In [10]:
fig_2 = px.line(
    issue_stats,
    x=plot_level,
    y="issue_length",
    color="issue",
    markers=False,
    title=f"Total Length of Sections for each Issue in UN Briefings between {start_date} and {end_date}"
)

fig_2.show()

In [11]:
issue_stats["proportion"] = (
    issue_stats["issue_length"] / issue_stats["total_length"]
).fillna(0)

fig_3 = px.line(
    issue_stats,
    x=plot_level,
    y="proportion",
    color="issue",
    markers=False,
    title=f"Proportion of Text for each Issue in UN Briefings between {start_date} and {end_date}"
)

fig_3.show()

In [12]:
fig_4 = px.bar(
    issue_counts,
    x="total_counts",
    y="issue",
    orientation="h",
    title=f"Total Counts of Issues in UN Briefings between {start_date} and {end_date}"
)

fig_4.update_yaxes(
    dtick=1,
    automargin=True,
    autorange="reversed",
)

fig_4.show()

**Comments on the Plot Results**

The issues with the highest presence are:
* ISSUE-20 Food security, hunger and malnutrition as a primary or cross-country issue
* ISSUE-18 Natural disasters and acute emergency response
* ISSUE-17 Climate change and long-term environmental issues
* ISSUE-1 Israel-Palestine conflict, Gaza and the Occupied Palestinian Territory
* ISSUE-21 Refugees, displacement and migration as a primary or cross-border issue
* ISSUE-12 Lebanon political and humanitarian crisis and Israel-Lebanon hostilities
* ISSUE-23 Sustainable development and development financing

Most of them are actually among the broader-scope issues, i.e., ISSUE-17 to ISSUE-24, as noted above.

If needed, we can focus on ISSUE-1 to ISSUE-16, whose trends may be more interesting and meaningful to analyze.